# P&ID Safety Copilot
**Gemma 4 Good Hackathon** | Theme: Climate & Environmental Impact

Fine-tuned Gemma 4 E4B that answers safety-critical questions about P&IDs:
isolation valves, component lists, single-point failure detection.
Runs fully offline on T4 GPU. GGUF 4-bit export for CPU-only deployment.

In [ ]:
# GPU compatibility check — P100 (sm_60) not supported by PyTorch 2.10+cu128
import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    if major < 7:
        name = torch.cuda.get_device_name()
        raise RuntimeError(
            f"\n{'='*60}\n"
            f"INCOMPATIBLE GPU: {name} (sm_{major}{minor})\n"
            f"PyTorch 2.10+cu128 requires sm_70+ (T4, V100, A100).\n"
            f"\nFix: Kaggle > Settings > Accelerator > select 'T4 GPU x1' > Save > Restart\n"
            f"{'='*60}"
        )
print(f"GPU OK: {torch.cuda.get_device_name()} sm_{torch.cuda.get_device_capability()[0]}{torch.cuda.get_device_capability()[1]}")

In [ ]:
# Cell 1 — install dependencies and clone repo (training data is inside the repo)
import subprocess, sys, os
subprocess.run(["pip", "install", "unsloth[kaggle-new]", "trl", "datasets", "networkx", "-q"], check=False)
subprocess.run(["git", "clone", "https://github.com/govindrathore27/gemma4-engineering-diagrams.git",
                "/kaggle/working/repo"], capture_output=True)
subprocess.run(["git", "-C", "/kaggle/working/repo", "pull"], capture_output=True)
sys.path.insert(0, "/kaggle/working/repo")

# Training data already in the repo — no external downloads needed
TRAIN_JSONL = "/kaggle/working/repo/project1_pid_copilot/data/train.jsonl"
EVAL_JSONL  = "/kaggle/working/repo/project1_pid_copilot/data/eval.jsonl"

import pathlib
n_train = sum(1 for _ in open(TRAIN_JSONL))
n_eval  = sum(1 for _ in open(EVAL_JSONL))
print(f"Training data ready: {n_train} train rows, {n_eval} eval rows")

In [ ]:
# Cell 2 — fine-tune Gemma 4 E4B with LoRA
os.environ["WANDB_DISABLED"] = "true"
from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="pid",
    data_path=TRAIN_JSONL,
    output_dir="/kaggle/working/pid_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Cell 3 — evaluate on held-out set
import json
from pathlib import Path
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/pid_adapter")
eval_rows = [json.loads(l) for l in open(EVAL_JSONL, encoding="utf-8")][:100]
predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected    = [r["output"] for r in eval_rows]
result = evaluate_batch(predictions, expected, metric="f1")
print(f"Token F1: {result['mean']:.3f}  ({len(eval_rows)} samples)")
Path("/kaggle/working/eval_results.txt").write_text(
    f"Token F1: {result['mean']:.3f}\nSamples: {len(eval_rows)}\n")

In [ ]:
# Cell 4 — export to GGUF 4-bit for offline / CPU deployment
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/pid_adapter",
    output_path="/kaggle/working/pid_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Cell 5 — demo inference
sample_graph = json.dumps({
    "nodes": [
        {"id": "pump1",       "label": "pump"},
        {"id": "valve1",      "label": "valve"},
        {"id": "valve2",      "label": "valve"},
        {"id": "vessel1",     "label": "vessel"},
        {"id": "instrument1", "label": "instrumentation"},
    ],
    "edges": [
        {"from": "pump1",       "to": "valve1"},
        {"from": "valve1",      "to": "vessel1"},
        {"from": "valve2",      "to": "vessel1"},
        {"from": "instrument1", "to": "vessel1"},
    ]
})

print("=== P&ID Safety Copilot Demo ===\n")
for q in [
    "List all valve components in this P&ID.",
    "What valves must be closed to isolate pump1?",
    "How many components of each type are in this P&ID?",
    "Identify single-point failures that could affect vessel1.",
]:
    print(f"Q: {q}\nA: {run(model, tokenizer, q, sample_graph)}\n")

## Impact

Industrial accidents at chemical plants, water treatment, and oil & gas installations
cause 40,000+ deaths/year. Engineers manually tracing P&IDs for isolation paths is
error-prone and slow. This model gives instant, auditable answers in natural language.

**Deployment:** GGUF 4-bit export runs via llama.cpp on any laptop — fully offline,
zero cloud dependency. Suitable for remote industrial sites with no internet.

**Training data:** 3,986 QA pairs auto-generated from PID2Graph (Zenodo 14803338, CC BY-SA)
via graph traversal (BFS isolation paths, single-point failure detection, component listing).